# Week 3, day 2 — Graded 02 SOLUTIONS: Combining   (tier 2 of 4)

Executed in the lab image (pandas 3.0.5) against the real files in `../data/`.
Every quoted number is what it actually printed.

Questions 2 and 8 are the ones to re-read — both are cases where doing the same
two steps in the other order gives a different answer.

Run this cell once to set up the data. Then work down the sheet.

In [ ]:
# Graded 02 — Combining. Run this once.
import numpy as np
import pandas as pd

orders = pd.read_csv("../data/orders_long.csv")
cust = pd.read_csv("../data/customers_messy.csv")
wide = pd.read_csv("../data/technology_wide.csv")

print("orders:", orders.shape, "| customers:", cust.shape, "| wide:", wide.shape)

TIER 2 — two or three steps

### Question 1

2012: Technology, then Furniture, then Office Supplies.

Filter, group, sort -- three steps, and the only trap is doing the filter
after the group, which would give you every year's totals.

In [ ]:
y = orders[orders["Year"] == 2012]
print(y.groupby("Category")["Sales"].sum().round(2)
       .sort_values(ascending=False).to_string())

### Question 2

Dedupe then strip -> **`411`** rows. Strip then dedupe -> **`400`**. A difference of **`11`**.

Same two operations, opposite order, eleven rows of difference.

De-duplicating first compares raw strings, so `'Bill Donatelli'` and
`'Bill Donatelli '` are different rows and both survive. Stripping first
makes them identical and the second one goes.

**Normalise, then de-duplicate.** The other order leaves near-duplicates
behind and reports a clean row count while doing it.

In [ ]:
a = cust.drop_duplicates().copy()
a["CustomerName"] = a["CustomerName"].str.strip()

b = cust.copy()
b["CustomerName"] = b["CustomerName"].str.strip()
b = b.drop_duplicates()

print("dedupe then strip:", len(a))
print("strip then dedupe:", len(b))
print("difference:       ", len(a) - len(b))

### Question 3

A `Region` x band table whose grand total is `1605576.22`, matching the raw total.

`observed=True` matters when grouping on a categorical, or Pandas keeps
every defined band including empty ones. `fill_value=0` is defensible here
because a region-band with no orders genuinely sold zero.

The reconciliation is the check that nothing fell outside the bins -- which
is only true because the top edge is `inf`.

In [ ]:
work = orders.copy()
work["Band"] = pd.cut(work["Sales"], bins=[0, 500, 5000, float("inf")],
                      labels=["small", "medium", "large"])
pt = work.pivot_table(index="Region", columns="Band", values="Sales",
                      aggfunc="sum", observed=True, fill_value=0)
print(pt.round(2).to_string())
print()
print("grand total:", round(pt.sum().sum(), 2),
      "| raw:", round(orders["Sales"].sum(), 2))

### Question 4

The wide file stacks to a `(32, 3)` long frame, and per-year Technology totals come out of a `groupby` that was impossible before.

`Region` has to become the index first, because `stack()` works on the
columns and `Region` is not one of the years.

That `groupby("Year")` is the whole argument for stacking: with years as
column *names* there is no `Year` column to group on, so the question could
not be asked at all.

In [ ]:
tidy = (wide.set_index("Region").stack()
             .rename_axis(["Region", "Year"]).reset_index(name="Sales"))
print("shape:", tidy.shape)
print()
print(tidy.groupby("Year")["Sales"].sum().round(2).to_string())

### Question 5

Pivot, add a row-wise total, sort by it -> Ontario, West, Atlantic, Prarie on top.

`sum(axis=1)` sums across the columns for each row. Adding it as a column
and sorting is the ordinary way to rank a wide table.

Worth noting that this total is computed from cells that were already
rounded to 2dp, so it can differ in the last penny from the total of the raw
column -- graded 03 Q7 is that effect on purpose.

In [ ]:
pt = orders.pivot_table(index="Region", columns="Year",
                        values="Sales", aggfunc="sum").round(2)
pt["Total"] = pt.sum(axis=1).round(2)
print(pt.sort_values("Total", ascending=False).head(4).to_string())

### Question 6

Margin per region, worst to best, as a percentage to 1dp.

Built from two pivots divided cell by cell, which works because both have
the same index.

Doing it the other way -- adding a per-row margin column and averaging it --
gives a different and usually wrong number, because it weights a small
order the same as a large one. The extra-practice sheet 05 Q7 has the two
side by side with opposite signs.

In [ ]:
p = orders.pivot_table(index="Region", values="Profit", aggfunc="sum")
s = orders.pivot_table(index="Region", values="Sales", aggfunc="sum")
margin = (p["Profit"] / s["Sales"] * 100).round(1).sort_values()
print(margin.to_string())

### Question 7

Per category: the count of IQR-flagged orders and the share of that category's revenue they carry.

Flagging then grouping keeps the outliers in the data where you can
measure them, instead of deleting them and reporting on what is left.

The revenue share is the number that matters. A category where 5% of orders
carry 50% of revenue behaves very differently from one where they carry 8%,
and the flag count alone does not distinguish them.

In [ ]:
work = orders.copy()
q1, q3 = work["Sales"].quantile(0.25), work["Sales"].quantile(0.75)
iqr = q3 - q1
work["Out"] = (work["Sales"] < q1 - 1.5 * iqr) | (work["Sales"] > q3 + 1.5 * iqr)

g = work.groupby("Category").apply(
    lambda d: pd.Series({
        "orders": len(d),
        "flagged": int(d["Out"].sum()),
        "pct_revenue": round(100 * d.loc[d["Out"], "Sales"].sum() / d["Sales"].sum(), 1),
    }), include_groups=False)
print(g.to_string())

### Question 8

Filter-then-mean and the pivot's mean are **identical**. Averaging the per-region *totals* gives **`50613.7`** against the true average order of **`1522.22`**.

Two of the three are the same operation written differently -- both average
the individual orders, so they agree exactly.

The third averages a different thing. `50613.7` is the average *regional
total*, not the average order, and it is 33 times larger. Both are real
numbers about 2012; only one answers 'what does a typical order look like'.

The general trap: when someone says 'the average', ask what the rows of the
thing being averaged are.

In [ ]:
a = orders[orders["Year"] == 2012].groupby("Region")["Sales"].mean().round(2)
b = orders.pivot_table(index="Region", columns="Year", values="Sales",
                       aggfunc="mean")[2012].round(2)
print("filter-then-mean vs pivot mean identical:", a.equals(b))
print()
print(a.to_string())
print()
totals = orders[orders["Year"] == 2012].groupby("Region")["Sales"].sum()
print("mean of the per-region TOTALS:", round(totals.mean(), 2))
print("mean of the individual ORDERS:", round(orders[orders["Year"] == 2012]["Sales"].mean(), 2))

### Question 9

Grouped `30` rows -> unstacked `32` cells -> stacked back **`32`** rows, of which **`2`** are `NaN`.

The round trip added two rows that were never in the data: Nunavut has no
Technology orders in 2009 or 2012, so `unstack` created the cells and
`stack` faithfully preserved them.

In pandas 3 `stack()` keeps `NaN` rows; older versions dropped them, so
this used to return 30. Anything you read online about `stack` dropping
missing values is describing the old behaviour.

`.dropna()` after the round trip restores the original exactly -- as a step
someone can read, rather than a default that changed between versions.

In [ ]:
tech = orders[orders["Category"] == "Technology"]
long1 = tech.groupby(["Region", "Year"])["Sales"].sum()
wide1 = long1.unstack()
long2 = wide1.stack()

print("grouped (long):   ", len(long1))
print("unstacked (cells):", wide1.size)
print("stacked back:     ", len(long2))
print("NaN rows in the round trip:", int(long2.isna().sum()))

### Question 10

The too-low bins total **`814021.73`** against the true `1605576.22` -- `96` rows unbinned, **no error**. Then `qcut(Year, q=10)` -> **raises** `ValueError: Bin edges must be unique`.

Half the revenue disappeared and nothing said so. `pd.cut` discards
anything outside its outermost edges, and `groupby` on the resulting
categorical skips the `NaN`s, so the report is internally consistent and
short by 49%.

The `qcut` in the second half stops you dead, because `Year` has 4 distinct
values and 10 quantiles need 11 distinct edges.

That contrast is the whole point of tier 3: the operation that raised cost
you ten seconds; the one that ran cost you a report.

In [ ]:
work = orders.copy()
work["Band"] = pd.cut(work["Sales"], bins=[0, 500, 5000],
                      labels=["small", "medium"])
print("banded total:", round(work.groupby("Band", observed=True)["Sales"].sum().sum(), 2))
print("real total:  ", round(orders["Sales"].sum(), 2))
print("rows unbinned:", work["Band"].isna().sum(), "-- no error was raised")
print()
print("distinct Year values:", orders["Year"].nunique(), "-- asking for 10 quantiles")
print(pd.qcut(orders["Year"], q=10))